# Plotly with IHS5 (Solutions)

## Setup

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

In [ ]:

DATA = Path('../../../data/0_raw/malawi/IHS 5 DATA sample')

print('Input folder:', DATA)

### Load source tables

In [ ]:
hh = pd.read_stata(DATA / 'hh_mod_a_filt.dta', convert_categoricals=True)
roster = pd.read_stata(DATA / 'HH_MOD_B.dta', convert_categoricals=True)
edu = pd.read_stata(
    DATA / 'HH_MOD_C.dta',
    convert_categoricals=True,
    columns=['case_id', 'PID', 'hh_c08', 'hh_c09']
)
cons = pd.read_stata(DATA / 'ihs5_consumption_aggregate.dta', convert_categoricals=True)

print('Loaded shapes:')
print('  hh    ', hh.shape)
print('  roster', roster.shape)
print('  edu   ', edu.shape)
print('  cons  ', cons.shape)

## Build final analysis table (shared prep)

In [ ]:
# Household size and head profile
hh_size = roster.groupby('case_id').size().rename('hh_size')
is_head = (
    (pd.to_numeric(roster['hh_b04'], errors='coerce') == 1)
    | (roster['hh_b04'].astype('string').str.strip().str.upper() == 'HEAD')
)

head = roster.loc[is_head, ['case_id', 'PID', 'hh_b05a', 'hh_b03']].copy()
head.columns = ['case_id', 'head_pid', 'head_age', 'head_sex']
head['head_age'] = pd.to_numeric(head['head_age'], errors='coerce')

head_edu = edu[['case_id', 'PID', 'hh_c08', 'hh_c09']].copy()
head_edu.columns = ['case_id', 'head_pid', 'head_education_proxy', 'head_qualification_code']

cons['pcrexpagg'] = cons['rexpaggpc'].copy()
cons_sel = cons[['case_id', 'rexpagg', 'pcrexpagg', 'poor']].copy()

In [ ]:
df = (
    hh
    .merge(hh_size, on='case_id', how='left')
    .merge(head, on='case_id', how='left')
    .merge(head_edu, on=['case_id', 'head_pid'], how='left')
    .merge(cons_sel, on='case_id', how='left')
)

df['urban_rural'] = df['reside']
for col in ['hh_size', 'head_age', 'rexpagg', 'pcrexpagg']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(f'Households: {len(df):,}')
df[['case_id', 'region', 'district', 'urban_rural', 'hh_size', 'pcrexpagg', 'poor']].head()

## 0) Histogram

In [ ]:
hist_df = df[['pcrexpagg', 'urban_rural']].dropna().copy()
hist_df['pcrexpagg_clip'] = hist_df['pcrexpagg'].clip(upper=hist_df['pcrexpagg'].quantile(0.99))

fig = px.histogram(
    hist_df,
    x='pcrexpagg_clip',
    color='urban_rural',
    nbins=50,
    barmode='overlay',
    opacity=0.6,
    labels={'pcrexpagg_clip': 'Per-capita consumption (clipped p99)', 'count': 'Households'},
    title='Histogram of per-capita consumption by urban/rural'
)
fig.update_layout(template='plotly_white')
fig.show()

**Result Interpretation**
Urban and rural distributions overlap but differ in center and spread; clipping at p99 improves visibility of the main mass.

## 1) Stacked bar chart

In [ ]:
stack_df = (
    df[['urban_rural', 'poor']]
    .dropna()
    .groupby(['urban_rural', 'poor'], as_index=False)
    .size()
)

fig = px.bar(
    stack_df,
    x='urban_rural',
    y='size',
    color='poor',
    barmode='stack',
    labels={'size': 'Households', 'urban_rural': 'Residence', 'poor': 'Poverty status'},
    title='Household counts by residence and poverty status'
)
fig.update_layout(template='plotly_white')
fig.show()

**Result Interpretation**
The composition of poor/non-poor households differs by residence type, which is easier to compare in stacked form.

## 2) Grouped bar chart

In [ ]:
group_df = (
    df[['region', 'urban_rural', 'pcrexpagg']]
    .dropna()
    .groupby(['region', 'urban_rural'], as_index=False)
    .agg(mean_pcrexpagg=('pcrexpagg', 'mean'))
)

fig = px.bar(
    group_df,
    x='region',
    y='mean_pcrexpagg',
    color='urban_rural',
    barmode='group',
    labels={'mean_pcrexpagg': 'Mean per-capita consumption', 'region': 'Region'},
    title='Mean per-capita consumption by region and residence'
)
fig.update_layout(template='plotly_white')
fig.show()

**Result Interpretation**
Grouping side-by-side shows within-region urban-rural gaps directly, without mixing totals.

## 3) Timeline chart

In [ ]:
time_df = df[['interviewDate', 'urban_rural', 'pcrexpagg']].dropna().copy()
time_df['interviewDate'] = pd.to_datetime(time_df['interviewDate'], errors='coerce')
time_df = time_df.dropna(subset=['interviewDate'])
time_df['month'] = time_df['interviewDate'].dt.to_period('M').dt.to_timestamp()

monthly = (
    time_df.groupby(['month', 'urban_rural'], as_index=False)
    .agg(mean_pcrexpagg=('pcrexpagg', 'mean'))
)

fig = px.line(
    monthly,
    x='month',
    y='mean_pcrexpagg',
    color='urban_rural',
    markers=True,
    labels={'month': 'Interview month', 'mean_pcrexpagg': 'Monthly mean pcrexpagg'},
    title='Monthly trend in per-capita consumption'
)
fig.update_layout(template='plotly_white')
fig.show()

**Result Interpretation**
The timeline captures month-to-month movement and whether urban/rural trends move in parallel.

## 4) Scatter

In [ ]:
scatter_df = df[['head_age', 'pcrexpagg', 'urban_rural', 'poor']].dropna().copy()
scatter_df = scatter_df[scatter_df['pcrexpagg'] <= scatter_df['pcrexpagg'].quantile(0.99)]
if len(scatter_df) > 4000:
    scatter_df = scatter_df.sample(4000, random_state=42)

fig = px.scatter(
    scatter_df,
    x='head_age',
    y='pcrexpagg',
    color='urban_rural',
    hover_data=['poor'],
    opacity=0.55,
    labels={'head_age': 'Head age', 'pcrexpagg': 'Per-capita consumption'},
    title='Head age vs per-capita consumption'
)
fig.update_layout(template='plotly_white')
fig.show()

**Result Interpretation**
The scatter highlights dispersion and potential non-linear patterns between head age and welfare.

## 4.1) Scatter + Regression Line

In [ ]:
reg_df = df[['head_age', 'pcrexpagg', 'urban_rural']].dropna().copy()
reg_df = reg_df[reg_df['pcrexpagg'] <= reg_df['pcrexpagg'].quantile(0.99)]
if len(reg_df) > 4000:
    reg_df = reg_df.sample(4000, random_state=42)

fig = px.scatter(
    reg_df,
    x='head_age',
    y='pcrexpagg',
    color='urban_rural',
    opacity=0.55,
    trendline='ols',
    trendline_scope='overall',
    trendline_color_override='#C44E52',
    labels={'head_age': 'Head age', 'pcrexpagg': 'Per-capita consumption'},
    title='Scatter with OLS regression line (head age)'
)
fig.update_layout(template='plotly_white')
fig.show()


**Result Interpretation**
The fitted line summarizes the average linear association; the wide cloud around it shows substantial household-level variation.

## 5) Bubble chart

In [ ]:
bubble_df = (
    df[['district', 'region', 'head_age', 'pcrexpagg']]
    .dropna()
    .groupby(['district', 'region'], as_index=False)
    .agg(
        avg_head_age=('head_age', 'mean'),
        avg_pcrexpagg=('pcrexpagg', 'mean'),
        n_households=('head_age', 'size')
    )
    .sort_values('n_households', ascending=False)
    .head(25)
)

fig = px.scatter(
    bubble_df,
    x='avg_head_age',
    y='avg_pcrexpagg',
    size='n_households',
    color='region',
    hover_name='district',
    size_max=60,
    labels={
        'avg_head_age': 'Average head age',
        'avg_pcrexpagg': 'Average per-capita consumption',
        'n_households': 'Households in district'
    },
    title='District profile bubbles (top 25 by sample size)'
)
fig.update_layout(template='plotly_white')
fig.show()

**Result Interpretation**
Bubble size adds sample context, so large and small districts are not interpreted with equal weight.

## 6) Sankey diagram (education -> poverty)

In [ ]:
sankey_df = df[['head_education_proxy', 'poor']].dropna().copy()
sankey_df['head_education_proxy'] = sankey_df['head_education_proxy'].astype('string').str.strip()
sankey_df['poor'] = sankey_df['poor'].astype('string').str.strip()

top_edu = sankey_df['head_education_proxy'].value_counts().head(8).index
sankey_df.loc[~sankey_df['head_education_proxy'].isin(top_edu), 'head_education_proxy'] = 'Other education'

flows = (
    sankey_df.groupby(['head_education_proxy', 'poor'], as_index=False)
    .size()
    .rename(columns={'size': 'value'})
)

left_nodes = sorted(flows['head_education_proxy'].unique())
right_nodes = sorted(flows['poor'].unique())
nodes = left_nodes + right_nodes
node_to_idx = {name: i for i, name in enumerate(nodes)}

source = flows['head_education_proxy'].map(node_to_idx).tolist()
target = flows['poor'].map(node_to_idx).tolist()
value = flows['value'].tolist()

fig = go.Figure(
    data=[
        go.Sankey(
            node=dict(
                pad=16,
                thickness=16,
                line=dict(color='black', width=0.4),
                label=nodes
            ),
            link=dict(source=source, target=target, value=value)
        )
    ]
)
fig.update_layout(title_text='Flow from head education proxy to poverty status', font_size=11)
fig.show()

**Result Interpretation**
Sankey makes compositional flow explicit, showing where large education groups end up across poverty categories.

## 7) Custom Template

In [ ]:
PALETTE = ["#7BB3B2", "#65A6BD", "#C997AF", "#B8B0D3", "#F4CF97", "#98B9A0", "#F6DECD"]
import plotly.graph_objects as go
import plotly.io as pio

nso_template = go.layout.Template()
nso_template.layout = go.Layout(
    font=dict(family='Arial', size=13, color='#333'),
    title_font=dict(size=16, color='#222'),
    plot_bgcolor='white',
    paper_bgcolor='white',
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=True, gridcolor='lightgray', gridwidth=0.5),
    colorway=PALETTE,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
    margin=dict(l=60, r=30, t=60, b=40)
)

pio.templates['nso'] = nso_template

custom_df = (
    df[['region', 'urban_rural', 'pcrexpagg']]
    .dropna()
    .groupby(['region', 'urban_rural'], as_index=False)
    .agg(mean_pcrexpagg=('pcrexpagg', 'mean'))
)

fig = px.bar(
    custom_df,
    x='region',
    y='mean_pcrexpagg',
    color='urban_rural',
    barmode='group',
    title='Grouped bar chart with NSO custom template',
    labels={'mean_pcrexpagg': 'Mean per-capita consumption'}
)
fig.update_layout(template='nso')
fig.show()

**Result Interpretation**
A custom template enforces consistent styling across figures, reducing repetitive layout code.

## 8) Pivot Tables + Pandas Formatting

In [ ]:
pivot_mean = df.pivot_table(
    values='pcrexpagg',
    index='region',
    columns='urban_rural',
    aggfunc='mean'
)

pivot_mean.style.format('{:,.0f}').set_caption(
    'Average Monthly Income by Region and Area Type (DKW)'
)

In [ ]:
tmp = df[['region', 'urban_rural', 'poor']].dropna().copy()
tmp['is_poor'] = (tmp['poor'].astype('string').str.strip().str.upper() == 'POOR').astype(float)

pivot_poor = tmp.pivot_table(
    values='is_poor',
    index='region',
    columns='urban_rural',
    aggfunc='mean'
)

pivot_poor.style.format('{:.1%}').background_gradient(cmap='Reds').set_caption(
    'Poverty Rate by Region and Area Type'
)

**Result Interpretation**
Formatted pivot tables make summaries presentation-ready while preserving reproducible computation.

## 9) Plotly Table Formatting

In [ ]:
table_df = pivot_mean.reset_index().copy()
val_cols = [c for c in table_df.columns if c != 'region']

cell_values = [table_df['region'].astype('string').tolist()]
for c in val_cols:
    cell_values.append(table_df[c].map(lambda v: f'{v:,.0f}' if pd.notna(v) else '').tolist())

fig = go.Figure(
    data=[
        go.Table(
            header=dict(
                values=['Region'] + [str(c) for c in val_cols],
                fill_color='#E8F1F0',
                align='left',
                font=dict(color='#222', size=12)
            ),
            cells=dict(
                values=cell_values,
                fill_color='white',
                align='left',
                height=28
            )
        )
    ]
)
fig.update_layout(title='Formatted Plotly table: mean pcrexpagg by region and area')
fig.show()

**Result Interpretation**
Plotly tables are useful when you want interactive, shareable tabular outputs in notebooks or dashboards.

## 10) Subplots

In [ ]:
from plotly.subplots import make_subplots

hist_vals = df['pcrexpagg'].dropna()
hist_vals = hist_vals[hist_vals <= hist_vals.quantile(0.99)]

bar_counts = (
    df[['urban_rural']]
    .dropna()
    .groupby('urban_rural', as_index=False)
    .size()
)

sc_df = df[['head_age', 'pcrexpagg', 'urban_rural']].dropna().copy()
sc_df = sc_df[sc_df['pcrexpagg'] <= sc_df['pcrexpagg'].quantile(0.99)]
if len(sc_df) > 3000:
    sc_df = sc_df.sample(3000, random_state=42)

box_df = df[['urban_rural', 'pcrexpagg']].dropna().copy()
box_df = box_df[box_df['pcrexpagg'] <= box_df['pcrexpagg'].quantile(0.99)]

fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=(
        'Histogram: pcrexpagg (clipped p99)',
        'Household counts by urban/rural',
        'Scatter: head_age vs pcrexpagg',
        'Box: pcrexpagg by urban/rural'
    )
)

fig.add_trace(
    go.Histogram(x=hist_vals, nbinsx=40, marker_color='#7BB3B2', name='pcrexpagg'),
    row=1,
    col=1
)
fig.add_trace(
    go.Bar(x=bar_counts['urban_rural'], y=bar_counts['size'], marker_color='#65A6BD', name='count'),
    row=1,
    col=2
)
fig.add_trace(
    go.Scatter(
        x=sc_df['head_age'], y=sc_df['pcrexpagg'], mode='markers',
        marker=dict(size=6, opacity=0.35, color='#C997AF'), name='households'
    ),
    row=2,
    col=1
)
for grp, color in [('URBAN', '#B8B0D3'), ('RURAL', '#F4CF97')]:
    vals = box_df.loc[box_df['urban_rural'].astype('string').str.upper() == grp, 'pcrexpagg']
    if len(vals) > 0:
        fig.add_trace(
            go.Box(y=vals, name=grp, marker_color=color, boxmean=True, showlegend=False),
            row=2,
            col=2
        )

fig.update_layout(
    template='plotly_white',
    height=760,
    width=1100,
    title='Multi-panel Plotly dashboard',
    margin=dict(l=60, r=30, t=70, b=50)
)
fig.show()

**Result Interpretation**
Subplots combine complementary views in one figure, making it easier to compare distribution, composition, and relationships together.